In [1]:
import Tensor as t
import numpy as np
import matplotlib.pyplot as plt
import Operations as o
import Compile as c
from sklearn.datasets import fetch_openml

We will stick to the row major order to align with numpy. This means our "dense" layers will be

$$\bold{Y} = \bold{X}\bold{W} + \bold{b}$$

Where $\bold{X}$ is the row vector in question. Might implement a technique called batching in the future.

In [2]:
# Fetch the MNIST dataset (this might take a minute to download)
mnist = fetch_openml('mnist_784', version=1, as_frame=False)

# Split into features (images) and labels
X, y = mnist["data"], mnist["target"]

print(f"Dataset shape: {X.shape}")

Dataset shape: (70000, 784)


In [3]:
def encode(val):
    z = np.zeros(10,dtype=np.float64)
    z[int(val)] += 1.0
    return z

y_cleaned = np.array([encode(k) for k in y])

x_cleaned = X.astype(np.float64) / 255


In [4]:
print(x_cleaned.shape)

(70000, 784)


In [5]:
print(y_cleaned.shape)

(70000, 10)


In [6]:
#current architecture: Dense(784 15) sAct softmax Dense(15 10) sAct softmax (done!)
#paramaters
w_1 = 100*np.random.random_sample(size=(784, 15))
b_1 = np.random.random_sample(size=(1,15))

w_2 = 100*np.random.random_sample(size=(15,10))
b_2 = np.random.random_sample(size=(1,10))

def pipeline(input):
    L_1 = o.copy(t.TensorNode(input,is_param=False)) @ t.TensorNode(w_1) + t.TensorNode(b_1)
    L_1 = (o.sAct(L_1))
    L_1 = L_1/(o.sum(L_1,axis=(1),keepDim=True)+1e-9)
    
    L_2 = L_1 @ t.TensorNode(w_2) + t.TensorNode(b_2)
    L_2 = o.sAct(L_2)
    L_2 = L_2/(o.sum(L_2,axis=(1),keepDim=True)+1e-9)
    return L_2


In [7]:
L_1 = (o.copy(t.TensorNode(x_cleaned[0],is_param=False))) @ t.TensorNode(w_1) + t.TensorNode(b_1)
L_1 = o.regularization(o.sAct(L_1),1e-9)

L_2 = L_1 @ t.TensorNode(w_2) + t.TensorNode(b_2)

L_2 = o.regularization(o.sAct(L_2),1e-9)

In [8]:
modelCompiler = c.Pipeline(L_2.compile())
#use model to actually get predictions and vector ouputs!
print(L_2.data)
print(y_cleaned[0])

[[0.10753131 0.08268615 0.1158903  0.09799956 0.10148432 0.113584
  0.1158829  0.08685349 0.09536069 0.08272727]]
[0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]


In [9]:
what = pipeline(x_cleaned)

In [10]:
modelCompiler.update_input(x_cleaned[3000])
print((L_2.data))

[[0.10773132 0.08274745 0.11781805 0.09813206 0.10107479 0.11307631
  0.11596366 0.08720489 0.09417548 0.082076  ]]


In [11]:
(what.data[3000])

array([0.10773132, 0.08274745, 0.11781805, 0.09813206, 0.10107479,
       0.11307631, 0.11596366, 0.08720489, 0.09417548, 0.082076  ])

In [12]:
loss = o.norm_squared(what - t.TensorNode(y_cleaned,is_param=False))
comp = (loss.compile())

In [26]:
comp[-2][1].gradient

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(784, 15))

In [25]:
prevLoss = loss.data
lr = 10.0
print(prevLoss)
for j in range(1):
    comp[0][0].temp_grad = np.ones_like(comp[0][0].temp_grad)
    #backwards prop
    for level in comp:
        for node in level:
            node.backprop()
    
print(prevLoss)

63011.75251493034
(15, 70000) (70000, 10) (10, 15)
(784, 70000) (70000, 15) (15, 784)
63011.75251493034


In [15]:
print(loss.data)

63011.75251493034


63108.25813250778
(15, 70000) (70000, 10) (10, 15)
(784, 70000) (70000, 15) (15, 784)
new lr 5.0
64634.11621406724

WE DID IT. Below is the accuracy:

In [16]:
correct = 0
for j in range(50_000):
    modelCompiler.update_input(np.array([x_cleaned[j]]))
    if(str(np.argmax(L_2.data)) == y[j]):
        correct+=1

print("Raw correct: " + str(correct))
print("accruacy " + str(correct/50_000))

print("test dataset:")
correct = 0
for j in range(50_001,70_000):
    modelCompiler.update_input(x_cleaned[j])
    if(str(np.argmax(L_2.data)) == y[j]):
        correct+=1

print("Raw correct: " + str(correct))
print("accruacy " + str(correct/20_000))


Raw correct: 5350
accruacy 0.107
test dataset:
Raw correct: 2209
accruacy 0.11045


Raw correct: 5101
accruacy 0.10202
test dataset:
Raw correct: 2039
accruacy 0.10195

Check out data.npz to import paramaters